# RMSD Integration and Pose Label Assignment

## Purpose
Merge per-pose GNINA scores (including RMSD to the crystal/reference structure)
into the MD-feature table, and assign binary classification labels based on
RMSD thresholds of 2.5 and 5.0 Angstroms.

## Label definition
- `label_2p5 = 1` if `rmsd_gninaA <= 2.5 Angstroms`, else 0 (strict criterion)
- `label_5p0 = 1` if `rmsd_gninaA <= 5.0 Angstroms`, else 0 (permissive criterion)

RMSD is computed as the heavy-atom RMSD between the docked pose and the
crystal/reference ligand structure using the Hungarian algorithm for atom
matching (see `gnina_docking_workflow.ipynb`).

## Input
- CSV table with docking scores, descriptors, and MD features (output of the MD feature generation step)
- `<PDB_ID>_gnina_scores.csv` (one per complex): GNINA score CSV produced by
  `gnina_docking_workflow.ipynb`, containing columns:
  `pose_index`, `vina_affinity`, `cnn_pose_score`, `cnn_affinity`, `rmsd_to_crystal_A` (heavy-atom RMSD)

## Output
- CSV table with `rmsd_gninaA`, `label_2p5`, `label_5p0` columns added immediately after `ligand_id`

## Run Order
1. Run notebook 02 (MD features) to generate the input feature table
2. Run docking workflow to generate `*_gnina_scores.csv` with RMSD column
3. Run this notebook

In [ ]:
# ================================================================
# Cell 1: Libraries
# ================================================================

import os
import re
import glob
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path


In [ ]:
# ================================================================
# Cell 2: Configuration
# ================================================================

# ── Configuration ──────────────────────────────────────────────────────────
# Update paths to match your local directory structure before running.
INPUT_CSV     = "/path/to/input.csv"
GNINA_DIR = "/path/to/analysis/rmsd"  # directory containing *_gnina_scores.csv files
OUT_CSV     = "/path/to/output.csv"

RMSD_THRESHOLDS = [2.5, 5.0]  # Å — label_2p5 and label_5p0
# ── End Configuration ──────────────────────────────────────────────────────


In [ ]:
# ================================================================
# Cell 3: Utilities
# ================================================================

# ── Utilities ───────────────────────────────────────────────────────────────
def norm_pose(s: str):
    """Normalize ligand_id to 'poseXX' format (CNN pose score order, GNINA default)."""
    if not isinstance(s, str):
        return None
    s = s.strip()
    # Match poseXX or pose-XX or pose XX (case-insensitive)
    m = re.match(r"(?:gnina[_\- ]?)?pose[_\- ]?(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"pose{int(m.group(1)):02d}"
    return s

def move_cols_after(df: pd.DataFrame, anchor: str, cols_to_move: list) -> pd.DataFrame:
    """Reorder columns so that cols_to_move appear immediately after anchor."""
    df = df.loc[:, ~df.columns.duplicated()].copy()
    if anchor not in df.columns:
        return df
    move = [c for c in cols_to_move if c in df.columns]
    rest = [c for c in df.columns if c not in move]
    i = rest.index(anchor) + 1
    return df.loc[:, rest[:i] + move + rest[i:]]


In [ ]:
# ================================================================
# Cell 4: Load feature table and GNINA scores
# ================================================================

# ── Load MD feature table ────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
if not {"pdb_id", "ligand_id"}.issubset(df.columns):
    raise RuntimeError("Input CSV must have 'pdb_id' and 'ligand_id' columns.")
print(f"Input rows: {len(df)}")

df["ligand_id_norm"] = df["ligand_id"].apply(norm_pose)

# ── Aggregate GNINA score CSVs ───────────────────────────────────────────────
# Each *_gnina_scores.csv is produced by gnina_docking_workflow.ipynb.
# Expected columns: pose_index (1-10), vina_affinity, cnn_pose_score, cnn_affinity, rmsd_to_crystal_A
rows = []
csv_files = sorted(glob.glob(os.path.join(GNINA_DIR, "*_gnina_scores.csv")))
if not csv_files:
    raise RuntimeError(f"No gnina score CSVs found in: {GNINA_DIR}")

print(f"Found {len(csv_files)} gnina score CSV(s)")

for path in csv_files:
    base = os.path.basename(path)
    m = re.match(r"([0-9A-Z]{4})_gnina_scores\.csv$", base, flags=re.IGNORECASE)
    if not m:
        continue
    pdb_id = m.group(1).upper()
    g = pd.read_csv(path)

    if "rmsd_to_crystal_A" not in g.columns:
        print(f"[WARN] {base}: 'rmsd_to_crystal_A' column not found — skipping")
        continue

    # Normalize pose identifier
    if "pose_index" in g.columns:
        g["ligand_id_norm"] = g["pose_index"].apply(lambda x: f"pose{int(x):02d}")
    elif "sdf_file" in g.columns:
        g["ligand_id_norm"] = (g["sdf_file"].astype(str)
                               .str.extract(r"(pose\d{2})", expand=False))
    else:
        print(f"[WARN] {base}: cannot determine pose identifier — skipping")
        continue

    g["pdb_id"] = pdb_id
    g = g.rename(columns={"rmsd_to_crystal_A": "rmsd_gninaA"})  # rename RMSD column only
    # vina_affinity, cnn_pose_score, cnn_affinity are already named correctly by gnina_docking_workflow.ipynb



    keep = ["pdb_id", "ligand_id_norm", "rmsd_gninaA",
            "vina_affinity", "cnn_pose_score", "cnn_affinity"]
    g = g[[c for c in keep if c in g.columns]].copy()
    for col in ["rmsd_gninaA", "vina_affinity", "cnn_pose_score", "cnn_affinity"]:
        if col in g.columns:
            g[col] = pd.to_numeric(g[col], errors="coerce")
    rows.append(g)

key_cols = ["pdb_id", "ligand_id_norm"]
g_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=key_cols)
if not g_all.empty:
    agg = {c: "mean" for c in g_all.columns if c not in key_cols}
    g_all = g_all.groupby(key_cols, as_index=False).agg(agg)
    print(f"GNINA score rows after aggregation: {len(g_all)}")


In [ ]:
# ================================================================
# Cell 5: Merge labels and summarize
# ================================================================

# ── Merge and label ──────────────────────────────────────────────────────────
merged = df.merge(g_all, on=key_cols, how="left", validate="m:1")

r = pd.to_numeric(merged.get("rmsd_gninaA"), errors="coerce")
for thr in RMSD_THRESHOLDS:
    tag = str(thr).replace(".", "p")
    merged[f"label_{tag}"] = ((r.notna()) & (r <= thr)).astype(int)

# Reorder: place RMSD, scores, and labels immediately after ligand_id
to_place = ["rmsd_gninaA", "vina_affinity", "cnn_pose_score", "cnn_affinity",
            "label_2p5", "label_5p0"]
out = move_cols_after(merged, anchor="ligand_id", cols_to_move=to_place)
out = out.drop(columns=["ligand_id_norm"], errors="ignore")
out = out.loc[:, ~out.columns.duplicated()]

# ── Summary ──────────────────────────────────────────────────────────────────
hits = out["rmsd_gninaA"].notna().sum()
print(f"\nRows with RMSD data:  {hits} / {len(out)}")
print(f"Positive (≤2.5 Å):    {int(out['label_2p5'].sum())}")
print(f"Positive (≤5.0 Å):    {int(out['label_5p0'].sum())}")

dups = [k for k, v in Counter(out.columns).items() if v > 1]
if dups:
    print(f"[WARN] Duplicate columns: {dups}")


In [ ]:
# ================================================================
# Cell 6: Save output
# ================================================================

# ── Save ─────────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUT_CSV) or ".", exist_ok=True)
out.to_csv(OUT_CSV, index=False)
print(f"\n[OK] Saved: {OUT_CSV}  shape={out.shape}")
